**Import packages and dependecies**

In [1]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


ERROR: file:///C:/Users/tcphan/OneDrive/Documents/Data%20Science%20Projects/topological does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import kagglehub

from tda.dimensionality_reduction import UMAP

### 1. Load in data inputs

The dataset below contains roughly 30,000 member records on various health conditions and risk factors, including:

- Demographics (e.g. age, gender)
- Chronic conditions (e.g. diabetes, hypertension)
- Biometric (e.g. glucose level, blood pressure)
- Medical risk factors (e.g. smoking, alcohol usage)
- Family medical history

For further information about the data, see [Healthcare Risk Factors Dataset](https://www.kaggle.com/datasets/abdallaahmed77/healthcare-risk-factors-dataset).

In [9]:
# Download data fram Kaggle
path = kagglehub.dataset_download(
    "abdallaahmed77/healthcare-risk-factors-dataset"
)

health_risk_factors_df = pl.read_csv(f"{path}/dirty_v3_path.csv")

# Show data
health_risk_factors_df.limit(50).show()

Age,Gender,Medical Condition,Glucose,Blood Pressure,BMI,Oxygen Saturation,LengthOfStay,Cholesterol,Triglycerides,HbA1c,Smoking,Alcohol,Physical Activity,Diet Score,Family History,Stress Level,Sleep Hours,random_notes,noise_col
f64,str,str,f64,f64,f64,f64,i64,f64,f64,f64,i64,i64,f64,f64,i64,f64,f64,str,f64
46.0,"""Male""","""Diabetes""",137.04,135.27,28.9,96.04,6,231.88,210.56,7.61,0,0,-0.2,3.54,0,5.07,6.05,"""lorem""",-137.057211
22.0,"""Male""","""Healthy""",71.58,113.27,26.29,97.54,2,165.57,129.41,4.91,0,0,8.12,5.9,0,5.87,7.72,"""ipsum""",-11.23061
50.0,null,"""Asthma""",95.24,null,22.53,90.31,2,214.94,165.35,5.6,0,0,5.01,4.65,1,3.09,4.82,"""ipsum""",98.331195
57.0,null,"""Obesity""",null,130.53,38.47,96.6,5,197.71,182.13,6.92,0,0,3.16,3.37,0,3.01,5.33,"""lorem""",44.187175
66.0,"""Female""","""Hypertension""",95.15,178.17,31.12,94.9,4,259.53,115.85,5.98,0,1,3.56,3.4,0,6.38,6.64,"""lorem""",44.831426


**List of features fields.**

In [10]:
features_list = [
    "Age",
    "Gender",
    "Medical Condition",
    "Glucose",
    "Blood Pressure",
    "BMI",
    "Oxygen Saturation",
    "LengthOfStay",
    "Cholesterol",
    "Triglycerides",
    "HbA1c",
    "Smoking",
    "Alcohol",
    "Physical Activity",
    "Diet Score",
    "Family History",
    "Stress Level",
    "Sleep Hours",
]

print(f"Total # of features: {len(features_list)}")

Total # of features: 18


### 2. Apply data preprocessing

**Remove columns if they have high missingness rate.**

In [11]:
# Initialize parameters
tot_n_rows = health_risk_factors_df.shape[0]
missing_threshold = (
    0.4  # Columns w/ missing rate less than or equal to threshold are kept
)

# Calculate the percent of missing in each column
missing_count_df = health_risk_factors_df.null_count() / tot_n_rows
missing_count_df = missing_count_df.unpivot(
    on=features_list, variable_name="Variable Name", value_name="p_missing"
)

# Remove columns from data
columns_failed_threshold_list = (
    missing_count_df.filter(pl.col("p_missing") > missing_threshold)
    .select("Variable Name")
    .to_series()
)
preprocessed_risk_factors_df = health_risk_factors_df.drop(*columns_failed_threshold_list)
features_list = [c for c in features_list if c not in columns_failed_threshold_list]
print(
    f"Total # of columns removed due to high missing rate: {len(columns_failed_threshold_list)}"
)

Total # of columns removed due to high missing rate: 0


**Apply one-hot encoding to categorical fields**

In [ ]:
# List of all string type columns
string_dtype_list = [
    name
    for name, dtype in preprocessed_risk_factors_df.select(features_list).schema.items()
    if dtype == pl.String
]

# Apply one-hot encoding
preprocessed_risk_factors_df = preprocessed_risk_factors_df.to_dummies(string_dtype_list)
ohe_vars_list = [col for col in preprocessed_risk_factors_df.columns for s in string_dtype_list if col.startswith(s)] 
preprocessed_risk_factors_df.select(ohe_vars_list).show()



Gender_Female,Gender_Male,Gender_null,Medical Condition_Arthritis,Medical Condition_Asthma,Medical Condition_Cancer,Medical Condition_Diabetes,Medical Condition_Healthy,Medical Condition_Hypertension,Medical Condition_Obesity,Medical Condition_null
u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
0,1,0,0,0,0,1,0,0,0,0
0,1,0,0,0,0,0,1,0,0,0
0,0,1,0,1,0,0,0,0,0,0
0,0,1,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,1,0,0
